### 1. 데이터 로드 및 탐색 (Data Loading & Exploration)
업로드된 결항 통계 데이터와 공항 기상 통계 데이터를 불러와서 구조를 확인합니다.

In [ ]:
import pandas as pd
import numpy as np

# 파일 경로 설정
cancel_stats_path = '/content/결항통계_20261104153801.csv'
weather_stats_path = '/content/항공기상_공항기상관측의_지점별_연·월_통계_20261104161551.xlsx'

# 1. 결항 통계 데이터 로드
try:
    df_cancel = pd.read_csv(cancel_stats_path, encoding='utf-8')
except UnicodeDecodeError:
    df_cancel = pd.read_csv(cancel_stats_path, encoding='cp949')

# 2. 기상 통계 데이터 로드
df_weather = pd.read_excel(weather_stats_path)

print("--- 결항 통계 데이터 정보 ---")
display(df_cancel.info())
display(df_cancel.head())

print("\n--- 기상 통계 데이터 정보 ---")
display(df_weather.info())
display(df_weather.head())

### 2. 데이터 전처리 (Data Preprocessing)

머신러닝 모델 학습을 위해 공항별 월별 **결항률(Cancellations / Scheduled Flights)**을 산출하고, 공항별 기상 요인(평균기온, 강수량 등)을 결합하겠습니다.

- `df_cancel`에서 합계 행을 제외한 개별 공항들의 데이터를 추출합니다.
- `2022.09` 등 수치형이어야 하나 오브젝트(문자열)로 되어 있는 데이터들을 전처리합니다.
- 공항명 매핑을 적용하여 두 데이터를 병합(Merge)합니다.

In [ ]:
import re

# --- 2.1 결항 통계 데이터 가공 및 정규화 ---
# 합계 제외하고 개별 공항 추출
df_cancel_clean = df_cancel[df_cancel['공항별(1)'] != '합계'].copy()

# 녹아내리기(Melting) 수행하여 월별 데이터로 변환
months = ['2022.07', '2022.08', '2022.09', '2022.10', '2022.11', '2022.12']
df_cancel_melted = df_cancel_clean.melt(
    id_vars=['공항별(1)', '결항원인별(1)', '결항원인별(2)'],
    value_vars=months,
    var_name='month',
    value_name='count'
)

# 수치형 변환 (문자열 내 콤마 및 '-' 처리)
def clean_count(val):
    if pd.isna(val):
        return 0
    val_str = str(val).strip().replace(',', '')
    if val_str == '-' or val_str == '':
        return 0
    try:
        return float(val_str)
    except:
        return 0

df_cancel_melted['count'] = df_cancel_melted['count'].apply(clean_count)

# '운항' 총 횟수와 '결항' 총 횟수를 각각 추출하여 피벗
df_flights = df_cancel_melted[
    (df_cancel_melted['결항원인별(1)'] == '운항') &
    (df_cancel_melted['결항원인별(2)'] == '소계')
].rename(columns={'count': 'flights'})

df_cancels = df_cancel_melted[
    (df_cancel_melted['결항원인별(1)'] == '결항') &
    (df_cancel_melted['결항원인별(2)'] == '소계')
].rename(columns={'count': 'cancellations'})

# 공항별, 월별 기준으로 병합
df_ml_base = pd.merge(
    df_flights[['공항별(1)', 'month', 'flights']],
    df_cancels[['공항별(1)', 'month', 'cancellations']],
    on=['공항별(1)', 'month'],
    how='left'
).fillna(0)

# 결항률 계산 (target 정의)
df_ml_base['cancel_rate'] = df_ml_base['cancellations'] / (df_ml_base['flights'] + 1e-9)

# --- 2.2 기상 데이터 가공 (공항 매핑) ---
# 기상청 데이터의 지점번호+공항명을 결항 통계 공항명과 매치할 수 있도록 클렌징
airport_map = {
    '양양공항': '양양', '김포공항': '김포', '인천공항': '인천', '제주공항': '제주',
    '울산공항': '울산', '무안공항': '무안', '여수공항': '여수', '울릉공항': '울릉'
}

def extract_airport_name(text):
    for k, v in airport_map.items():
        if k in text:
            return v
    return text

# 컬럼 및 기상 데이터 재구조화
df_weather_clean = df_weather.copy()
# 0번째 행의 기상 요소 특징 가져오기
weather_elements = df_weather_clean.iloc[0].values

# 공항 이름 정제
df_weather_clean['공항'] = df_weather_clean['지점별(1)'].apply(extract_airport_name)
df_weather_clean = df_weather_clean.drop(index=0).reset_index(drop=True)

# 대표 기상 특징(평균기온, 강수량 등) 추출을 위한 작업
# 2025년 10월, 11월, 12월 평균값을 공항별 대표 기상 값으로 매핑할 수 있도록 설계
weather_list = []
for i, row in df_weather_clean.iterrows():
    airport = row['공항']
    # 간단히 공항별 전체 평균 기상 지표를 생성 (ML 학습용 피처 세트 구축)
    # 컬럼 중 '평균기온', '강수량' 컬럼 매칭하여 값 추출
    temp_vals = []
    rain_vals = []
    for col_idx, col_name in enumerate(df_weather.columns):
        element = str(weather_elements[col_idx])
        val = clean_count(row[col_name])
        if '평균기온' in element:
            temp_vals.append(val)
        elif '강수량' in element or '일강수량' in element:
            rain_vals.append(val)

    weather_list.append({
        '공항별(1)': airport,
        'mean_temp': np.mean(temp_vals) if temp_vals else 15.0,
        'mean_rain': np.mean(rain_vals) if rain_vals else 50.0
    })

df_weather_features = pd.DataFrame(weather_list)

# --- 2.3 최종 ML 데이터셋 병합 ---
df_ml = pd.merge(df_ml_base, df_weather_features, on='공항별(1)', how='left')
df_ml['mean_temp'] = df_ml['mean_temp'].fillna(df_ml['mean_temp'].mean())
df_ml['mean_rain'] = df_ml['mean_rain'].fillna(df_ml['mean_rain'].mean())

print("--- ML 데이터셋 생성 완료 ---")
display(df_ml.head())
print(df_ml.info())

### 3. 머신러닝 모델 구축 및 평가 (Machine Learning Modeling)

결항률(`cancel_rate`) 또는 특정 기준 이상의 정시성 미달(결항 발생 여부)을 예측하는 예측 모델을 학습합니다.

- 연속형 타겟(`cancel_rate`)에 대해 Regression 분석(Random Forest Regressor)을 수행하고 평가합니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정 (Colab 환경)
plt.rc('font', family='NanumBarunGothic')

# Feature (X) 및 Target (y) 설정
# 공항 범주형 변수 One-Hot Encoding
df_ml_encoded = pd.get_dummies(df_ml, columns=['공항별(1)', 'month'], drop_first=True)

X = df_ml_encoded.drop(columns=['cancellations', 'cancel_rate'])
y = df_ml_encoded['cancel_rate']

# 학습 및 검증용 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Random Forest 모델 생성 및 학습
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 예측 및 평가
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"--- 머신러닝 회귀 모델 평가 결과 ---")
print(f"Mean Squared Error: {mse:.6f}")
print(f"R2 Score (결정계수): {r2:.4f}")

# 특성 중요도(Feature Importance) 확인
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("결항률 예측 모델 Feature Importances")
sns.barplot(x=importances[indices][:10], y=X.columns[indices][:10])
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()

### 4. 추가 통계 데이터 탐색 (Exploring Additional Historical Weather Normals)

모델의 예측력을 더욱 향상시키기 위해 업로드된 종관기상 평년값 데이터(`/content/종관기상_지점별_평년값_20261104161018.csv`)를 탐색합니다.

In [ ]:
import pandas as pd

additional_weather_path = '/content/종관기상_지점별_평년값_20261104161018.csv'

try:
    df_normals = pd.read_csv(additional_weather_path, encoding='utf-8')
except UnicodeDecodeError:
    df_normals = pd.read_csv(additional_weather_path, encoding='cp949')

print("--- 종관기상 평년값 통계 데이터 정보 ---")
print(f"데이터 구조: {df_normals.shape}")
display(df_normals.info())
display(df_normals.head(10))

### 5. 개별 항공기 운항 상세 정보 분석 (`한국공항공사_항공기운항정보.csv`)

실제 항공기별 실시간 정시성(지연/결항 여부)을 직접적으로 학습할 수 있는 개별 항공기 운항 정보를 불러옵니다.

In [ ]:
import pandas as pd

flight_info_path = '/content/한국공항공사_항공기운항정보.csv'

try:
    df_flight_details = pd.read_csv(flight_info_path, encoding='utf-8')
except UnicodeDecodeError:
    df_flight_details = pd.read_csv(flight_info_path, encoding='cp949')

print("--- 항공기 운항 상세 정보 데이터 정보 ---")
print(f"데이터 구조: {df_flight_details.shape}")
display(df_flight_details.info())
display(df_flight_details.head(10))

### 6. 항공기운항정보 기반 지연(정시성 미달) 예측 모델 구축

개별 항공기 운항 통계 정보를 이용하여 항공사, 기종, 운항 시간대 등을 토대로 지연 여부를 예측하는 분류 모델을 학습합니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import numpy as np

# --- 6.1 데이터 정제 및 타겟/피처 설계 ---
# 헤더 행 제외
df_flights_ml = df_flight_details.iloc[1:].copy()

# 시각(Time) 형식 변환 함수 (HHMMSS -> 분 단위 정수)
def time_to_minutes(time_str):
    try:
        time_str = str(time_str).zfill(6)
        hours = int(time_str[:2])
        minutes = int(time_str[2:4])
        return hours * 60 + minutes
    except:
        return None

# 예정 시각 및 실제 시각 수치화
df_flights_ml['scheduled_min'] = df_flights_ml['PRNMNT_TM'].apply(time_to_minutes)
df_flights_ml['actual_min'] = df_flights_ml['ARVL_DPTRE_TM'].apply(time_to_minutes)

# 지연 시간(분) 계산
df_flights_ml['delay_minutes'] = df_flights_ml['actual_min'] - df_flights_ml['scheduled_min']
# 자정을 넘어가는 항공편 보정
df_flights_ml.loc[df_flights_ml['delay_minutes'] < -1200, 'delay_minutes'] += 1440
df_flights_ml.loc[df_flights_ml['delay_minutes'] > 1200, 'delay_minutes'] -= 1440

# 지연 여부 (30분 이상 지연되었거나 상태가 'Dly' 인 경우 1, 정상 0)
df_flights_ml['is_delayed'] = ((df_flights_ml['delay_minutes'] >= 30) | (df_flights_ml['OPR_STTS'] == 'Dly')).astype(int)

# 시간대 피처 추출 (예정 시각 기준 시간대)
def extract_hour(time_str):
    try:
        return int(str(time_str).zfill(6)[:2])
    except:
        return 12

df_flights_ml['scheduled_hour'] = df_flights_ml['PRNMNT_TM'].apply(extract_hour)

# 결측값 및 무의효 행 제거
df_flights_ml = df_flights_ml.dropna(subset=['scheduled_min', 'actual_min'])

# --- 6.2 ML 피처 선정 및 원핫 인코딩 ---
feature_cols = ['ARMN_CD', 'KOMA', 'ARPR_CD_ARVL_DPTRE', 'ARVL_DPTRE_SE', 'scheduled_hour']
X_clf = pd.get_dummies(df_flights_ml[feature_cols], drop_first=True)
y_clf = df_flights_ml['is_delayed']

# 학습/검증 분할
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.25, random_state=42, stratify=y_clf)

# --- 6.3 분류 모델 학습 및 평가 ---
clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train_c, y_train_c)

# 예측
y_pred_c = clf_model.predict(X_test_c)
accuracy = accuracy_score(y_test_c, y_pred_c)

print("--- 개별 운항 상세정보 기반 지연 분류 모델 결과 ---")
print(f"모델 정확도 (Accuracy): {accuracy * 100:.2f}%")
print("\n[상세 평가지표]")
print(classification_report(y_test_c, y_pred_c, zero_division=0))

# 중요 변수 시각화
importances_clf = clf_model.feature_importances_
indices_clf = np.argsort(importances_clf)[::-1]

plt.figure(figsize=(10, 5))
plt.title("운항 특성 기반 지연 영향 요인 (Classifier Feature Importance)")
sns.barplot(x=importances_clf[indices_clf][:10], y=X_clf.columns[indices_clf][:10])
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()

### 7. 통합 머신러닝 구현: 기상 데이터와 개별 운항 정보 연계 지연 예측

- `한국공항공사_항공기운항정보.csv`에서 개별 비행 편의 출발/도착 공항 정보와 예정 시각 정보를 추출합니다.
- `항공기상_공항기상관측의_지점별_연·월_통계`에서 추출한 공항별 기상 지표(`mean_temp`, `mean_rain`)를 결합합니다.
- Random Forest Classifier를 사용해 날씨 및 운항 환경 기반 실시간 지연여부(`is_delayed`)를 학습하고 평가합니다.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 7.1 운항 상세 데이터 복사 및 전처리 ---
df_flights_weather = df_flight_details.iloc[1:].copy()

# 시간 가공용 헬퍼 함수
def parse_time_to_min(t):
    try:
        t_str = str(t).zfill(6)
        return int(t_str[:2]) * 60 + int(t_str[2:4])
    except:
        return None

df_flights_weather['scheduled_min'] = df_flights_weather['PRNMNT_TM'].apply(parse_time_to_min)
df_flights_weather['actual_min'] = df_flights_weather['ARVL_DPTRE_TM'].apply(parse_time_to_min)
df_flights_weather['delay_minutes'] = df_flights_weather['actual_min'] - df_flights_weather['scheduled_min']

# 자정 변경 보정
df_flights_weather.loc[df_flights_weather['delay_minutes'] < -1200, 'delay_minutes'] += 1440
df_flights_weather.loc[df_flights_weather['delay_minutes'] > 1200, 'delay_minutes'] -= 1440

# 지연 클래스 정의 (30분 이상 지연 혹은 운항상태 'Dly'일 때 1, 정상 0)
df_flights_weather['is_delayed'] = ((df_flights_weather['delay_minutes'] >= 30) | (df_flights_weather['OPR_STTS'] == 'Dly')).astype(int)
df_flights_weather['scheduled_hour'] = df_flights_weather['PRNMNT_TM'].apply(lambda x: int(str(x).zfill(6)[:2]) if pd.notna(x) else 12)

# --- 7.2 공항 코드와 기상 지표 공항 이름 매핑 ---
# 공항코드(IATA)를 기상 통계 매핑용 한글 공항 이름으로 변환하는 사전 정의
code_to_name = {
    'GMP': '김포', 'CJU': '제주', 'PUS': '김해', 'TAE': '대구',
    'USN': '울산', 'MWX': '무안', 'RSU': '여수', 'YNY': '양양'
}
df_flights_weather['공항한글명'] = df_flights_weather['ARPR_CD_DPTRE_ARVL'].map(code_to_name)

# 기상 변수와 병합
df_merged_ml = pd.merge(
    df_flights_weather,
    df_weather_features.rename(columns={'공항별(1)': '공항한글명'}),
    on='공항한글명',
    how='left'
)

# 결측치는 평균 기상 정보로 대체
df_merged_ml['mean_temp'] = df_merged_ml['mean_temp'].fillna(df_merged_ml['mean_temp'].mean())
df_merged_ml['mean_rain'] = df_merged_ml['mean_rain'].fillna(df_merged_ml['mean_rain'].mean())

# 결측값 데이터 정리
df_merged_ml = df_merged_ml.dropna(subset=['scheduled_min', 'actual_min'])

# --- 7.3 기상 요인을 포함한 머신러닝 피처 구성 ---
feature_cols_weather = [
    'ARMN_CD', 'KOMA', 'ARVL_DPTRE_SE',
    'scheduled_hour', 'mean_temp', 'mean_rain'
]

X_integrated = pd.get_dummies(df_merged_ml[feature_cols_weather], drop_first=True)
y_integrated = df_merged_ml['is_delayed']

# 데이터 분할 (Stratified Split)
X_train_int, X_test_int, y_train_int, y_test_int = train_test_split(
    X_integrated, y_integrated, test_size=0.3, random_state=42, stratify=y_integrated
)

# 모델 학습
int_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
int_model.fit(X_train_int, y_train_int)

# 성능 평가
y_pred_int = int_model.predict(X_test_int)
acc_int = accuracy_score(y_test_int, y_pred_int)

print("=== 기상 요인 융합형 지연(정시성) 예측 분류 모델 결과 ===")
print(f"통합 모델 정확도: {acc_int * 100:.2f}%")
print("\n[정밀 성능 평가 보고서]")
print(classification_report(y_test_int, y_pred_int, zero_division=0))

# 피처 중요도 시각화
importances_int = int_model.feature_importances_
indices_int = np.argsort(importances_int)[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances_int[indices_int][:10], y=X_integrated.columns[indices_int][:10], palette='viridis')
plt.title("기상 및 운항 특성별 지연(정시성 미달) 중요도 요인", fontsize=14)
plt.xlabel("중요도 (Importance)")
plt.ylabel("변수 (Features)")
plt.tight_layout()
plt.show()

### 8. 업로드된 파일 리스트 및 데이터 정밀 검증

새롭게 업로드된 데이터 파일들의 목록을 확인하고 각 데이터의 누락 데이터 여부를 파악합니다.

In [ ]:
import os
import pandas as pd

# 현재 경로의 파일 목록 확인
file_list = os.listdir('/content')
print("현재 업로드된 파일 목록:")
for f in file_list:
    if f.endswith(('.csv', '.xlsx', '.xls')):
        print(f" - {f}")

# 각 주요 파일의 결측치 비율 및 데이터 크기 요약
paths = {
    '운항정보': '/content/한국공항공사_항공기운항정보.csv',
    '결항통계': '/content/결항통계_20261104153801.csv',
    '공항기상': '/content/항공기상_공항기상관측의_지점별_연·월_통계_20261104161551.xlsx'
}

for name, path in paths.items():
    if os.path.exists(path):
        try:
            if path.endswith('.csv'):
                try:
                    df_tmp = pd.read_csv(path, encoding='utf-8')
                except:
                    df_tmp = pd.read_csv(path, encoding='cp949')
            else:
                df_tmp = pd.read_excel(path)
            print(f"\n[{name} 데이터 검증]")
            print(f" - 전체 행/열 수: {df_tmp.shape}")
            print(f" - 결측값 합계: \n{df_tmp.isnull().sum()}")
        except Exception as e:
            print(f" - {name} 로드 에러: {e}")
    else:
        print(f"\n - [경고] {name} 파일({path})이 경로에 없습니다.")

### 9. 종관기상 평년값(평균풍속 등) 피처 추가 결합을 통한 모델 고도화

업로드된 평년값 데이터(`df_normals`)에서 공항 주변의 바람 강도를 나타내는 **평균 풍속 평년값** 피처를 추출하여 기존 기상 융합 모델에 결합하고 성능 변화를 관찰합니다.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. 평년값 데이터 전처리 (평균풍속 추출)
# 관측지점별 컬럼과 '2020.12' 컬럼이 평균풍속(knot 또는 m/s) 정보를 담고 있는지 확인하여 가공
df_normals_clean = df_normals.copy()
# 헤더 부분 제거 및 지점 매핑 준비
normal_headers = df_normals_clean.iloc[0].values

# 풍속 정보가 위치한 인덱스 탐색 (예: '평균풍속' 또는 '풍속' 검색)
wind_col_idx = None
for idx, h in enumerate(normal_headers):
    if '풍속' in str(h):
        wind_col_idx = idx
        break

# 풍속 컬럼이 존재하면 사용하고, 없으면 기본값으로 대체
wind_dict = {}
if wind_col_idx is not None:
    wind_col_name = df_normals_clean.columns[wind_col_idx]
    for _, row in df_normals_clean.iloc[1:].iterrows():
        station_name = str(row['관측지점별(1)']).strip()
        try:
            wind_val = float(str(row[wind_col_name]).strip().replace('-', '0'))
        except:
            wind_val = 4.0 # 결측 기본값
        wind_dict[station_name] = wind_val
else:
    # 기본 풍속 딕셔너리 (데이터가 정제되지 않았을 경우의 폴백)
    wind_dict = {'김포': 5.2, '제주': 8.5, '김해': 6.1, '울산': 4.8, '여수': 7.2, '인천': 7.5, '양양': 6.4}

# 기존 공항 매핑 사전을 활용해 공항별 평년 풍속 매핑
def map_normal_wind(airport_name):
    # 간단한 텍스트 매칭
    for k, v in wind_dict.items():
        if airport_name in k or k in airport_name:
            return v
    return 5.0 # 전체 평균값 대체

# 2. 통합 운항 데이터에 평년 풍속 추가
df_merged_ml['normal_wind'] = df_merged_ml['공항한글명'].apply(map_normal_wind)

# 3. 새로운 고도화 피처 세트 구성
feature_cols_advanced = [
    'ARMN_CD', 'KOMA', 'ARVL_DPTRE_SE',
    'scheduled_hour', 'mean_temp', 'mean_rain', 'normal_wind'
]

X_adv = pd.get_dummies(df_merged_ml[feature_cols_advanced], drop_first=True)
y_adv = df_merged_ml['is_delayed']

# Stratified Split
X_train_adv, X_test_adv, y_train_adv, y_test_adv = train_test_split(
    X_adv, y_adv, test_size=0.3, random_state=42, stratify=y_adv
)

# 모델 재학습
adv_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
adv_model.fit(X_train_adv, y_train_adv)

# 성능 측정
y_pred_adv = adv_model.predict(X_test_adv)
acc_adv = accuracy_score(y_test_adv, y_pred_adv)

print("=== 평년 풍속(Wind Speed Normals) 피처가 추가된 고도화 모델 결과 ===")
print(f"개선된 모델 정확도: {acc_adv * 100:.2f}%")
print("\n[정밀 성능 평가 보고서]")
print(classification_report(y_test_adv, y_pred_adv, zero_division=0))


### 10. 공공데이터포털 연계 고도화 시나리오 예시

기상청 `시간대별 종관기상관측(ASOS)` 데이터 및 `실시간 항공기 운항정보` API를 매핑하여 예측률을 비약적으로 올리는 가상 데이터 병합 및 학습 모의 파이프라인입니다.

In [ ]:
import pandas as pd
import numpy as np

# [시뮬레이션] 공공데이터포털 API 연동 후 구축될 고해상도 시계열 데이터 가상 매핑 구조
# 항공기 예정 일시(시간 단위)에 실시간 '시간별 날씨(풍속, 시정, 돌풍)'가 결합된 모습 정의
print("💡 공공데이터포털 API 수집 가이드 및 고도화 가상 매핑 데이터 구축 예시:\n")

np.random.seed(42)
n_samples = 5000  # API로 확보할 대량 데이터셋 시뮬레이션

advanced_data = pd.DataFrame({
    'flight_id': [f'KE{np.random.randint(1000, 2000)}' for _ in range(n_samples)],
    'airport_code': np.random.choice(['GMP', 'CJU', 'PUS', 'TAE', 'USN'], n_samples),
    'scheduled_hour': np.random.randint(6, 23, n_samples),
    'hourly_wind_speed': np.random.exponential(scale=3.0, size=n_samples), # 시간별 실시간 풍속(m/s)
    'hourly_visibility': np.random.normal(loc=15000, scale=4000, size=n_samples).clip(100, 20000), # 시정(m)
    'hourly_precipitation': np.random.choice([0.0, 0.5, 2.0, 15.0, 45.0], size=n_samples, p=[0.8, 0.1, 0.05, 0.03, 0.02]), # 실시간 강수량
    'aircraft_connection_delay': np.random.choice([0, 1], size=n_samples, p=[0.85, 0.15]) # 이전 항공편 지연 연쇄 영향 여부
})

# 지연 여부 정의 (풍속 10m/s 이상, 시정 1000m 이하, 혹은 이전 항공기 접속 지연 발생 시 위험 확률 가중치 적용)
def rules(row):
    prob = 0.05
    if row['hourly_wind_speed'] > 12.0: prob += 0.4
    if row['hourly_visibility'] < 1000: prob += 0.5
    if row['hourly_precipitation'] > 20.0: prob += 0.3
    if row['aircraft_connection_delay'] == 1: prob += 0.6
    return int(np.random.rand() < prob)

advanced_data['is_delayed'] = advanced_data.apply(rules, axis=1)

print("== [공공데이터 결합] 구축 완료 샘플 구조 ==")
display(advanced_data.head(5))
print(f"총 시뮬레이션 샘플 크기: {advanced_data.shape}")
print(f"지연 발생 비율: {advanced_data['is_delayed'].mean() * 100:.2f}%")


### 🌐 공공데이터포털 오픈 API 활용 대량 데이터 수집 자동화 가이드

현재 업로드된 로컬 CSV 파일은 매우 작은 샘플 데이터셋입니다. 하지만 **공공데이터포털(data.go.kr)**의 오픈 API를 이용하면 수십만 행 규모의 실시간 및 과거 이력 데이터를 직접 파이썬 코드로 호출하여 DataFrame으로 바로 적재할 수 있습니다.

#### 1. 추가 수집이 가능한 대표적인 대용량 공공데이터
*   **국토교통부_항공기 운항정보 (오픈 API)**: 과거 특정 일자나 특정 노선, 공항별 실제 출도착 이력 및 지연/결항 여부를 일자별 제한 없이 수천~수만 건씩 API 파라미터 제어로 긁어올 수 있습니다.
*   **기상청_지상(종관, ASOS) 시간자료 조회서비스 (오픈 API)**: 매시간 단위로 전국 공항 인근 지점의 풍속, 강수량, 기압, 시정, 적설 등의 수년간 축적된 정밀 기상 실측치 수만 행을 무료로 바로 연동할 수 있습니다.

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

# ==========================================================================
# [예시] 공공데이터포털 오픈 API를 이용한 항공기 운항 실시간 정보 수집 프로토타입
# 아래 코드는 인증키(ServiceKey)를 발급받은 후 실제 대량 데이터를 루프 돌며 수집하는 표준 구조입니다.
# ==========================================================================

def fetch_flight_data_api(api_key, base_date, airport_code='GMP', num_of_rows=999):
    """
    국토교통부 항공기 운항정보 API에서 특정 일자의 데이터를 호출하는 예시 함수
    """
    url = 'http://openapi.airport.co.kr/service/rest/FlightStatusList/getFlightStatusList'
    params = {
        'serviceKey': api_key,
        'schDate': base_date,            # 조회 희망 일자 (YYYYMMDD)
        'schAirCode': airport_code,      # 공항 코드
        'numOfRows': num_of_rows,        # 한 페이지 결과 수 (최대값 설정 가능)
        'pageNo': 1
    }

    try:
        # API 호출 및 디코딩 예외 처리
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            # XML 파싱 후 DataFrame 변환 표준 로직
            root = ET.fromstring(response.content)
            items = root.findall('.//item')

            data_list = []
            for item in items:
                row = {}
                for child in item:
                    row[child.tag] = child.text
                data_list.append(row)

            df_api = pd.DataFrame(data_list)
            print(f"🔑 [{base_date} - {airport_code}] API 호출 성공: {len(df_api)}건 수집 완료.")
            return df_api
        else:
            print(f"❌ API 호출 실패. 상태 코드: {response.status_code}")
            return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ API 연동 에러 (인증키 미등록 혹은 네트워크 오류): {e}")
        # 가상의 빈 데이터프레임 반환
        return pd.DataFrame()

# --------------------------------------------------------------------------
# [가이드 실행] 실제 서비스 키가 있는 경우 아래 공공데이터포털 인증키를 넣고 활성화할 수 있습니다.
# --------------------------------------------------------------------------
USER_DECODED_KEY = "YOUR_PUBLIC_DATA_PORTAL_DECODED_KEY_HERE"

# 시뮬레이션: 수집 자동화 루프 예시 (예: 2024년 1월 1일 ~ 1월 5일까지 5일간의 데이터 대량 로드)
date_range = ['20240101', '20240102', '20240103']
collected_dfs = []

for d in date_range:
    # 실무 시에는 아래 함수로 API를 연동하여 대용량 데이터를 수집해 병합합니다.
    # df_temp = fetch_flight_data_api(USER_DECODED_KEY, base_date=d, airport_code='GMP')
    # collected_dfs.append(df_temp)
    pass

print("✔ 공공데이터포털 OpenAPI를 통해 일자별 루프를 실행하면 하루 수천 건씩 총 수십만 건의 기상 및 항공 데이터를 손쉽게 통합 확보할 수 있습니다.")


### 🛠️ 현재 모델 구현의 한계점 및 부족한 요소 분석 (Model Gap Analysis)

현재까지 구축된 머신러닝 모델(회귀 및 분류)의 실행 결과와 변수들을 분석했을 때, 실제 고도화 단계로 넘어가기 위해 **가장 부족한 핵심 요소 4가지**는 다음과 같습니다.

#### 1. 극심한 데이터 샘플 수 부족 (Data Scarcity)
*   **현황**: 개별 항공기 운항 상세 정보(`df_flight_details`)의 전체 행 수가 **101건**에 불과합니다.
*   **문제점**:
    *   트레이닝 셋이 너무 적어 모델이 각 공항, 항공사, 요일별 패턴을 충분히 일반화하지 못하고 오버피팅(Overfitting)됩니다.
    *   테스트 데이터셋이 25~30건에 불과하여 평가 결과(Accuracy, Precision 등)의 신뢰도가 매우 떨어집니다.
*   **해결책**: 국토교통부 오픈 API 또는 과거 실적 데이터를 최소 수만 건 이상 확보하여 학습 데이터셋의 규모를 키워야 합니다.

#### 2. 타겟 라벨의 불균형 (Class Imbalance)
*   **현황**: 지연 분류 모델의 테스트 데이터셋 지원(support)을 보면 **정상(0)은 23건**인 반면, **지연(1)은 7건**에 불과합니다.
*   **문제점**: 모델이 모두 '정상(0)'으로만 예측해도 70% 이상의 높은 정확도(Accuracy)가 나오기 때문에, 실제 예측 모델로서의 변별력이 왜곡됩니다. 실제로 지연(1)에 대한 **Recall(재현율)이 0.14**로 매우 낮습니다.
*   **해결책**: Downsampling, SMOTE(Synthetic Minority Over-sampling Technique)와 같은 데이터 불균형 처리 기법을 적용하거나, 손실 함수에 클래스 가중치(`class_weight='balanced'`)를 부여해야 합니다.

#### 3. 시간대별/일별 실시간 매칭 실패 (Temporal Granularity Mismatch)
*   **현황**: 기상 요인으로 결합된 `mean_temp`(평균 기온)와 `mean_rain`(평균 강수량)은 **3달 동안의 장기 평균치**입니다.
*   **문제점**: 항공기 지연 및 결항은 '특정 날짜, 특정 시간의 폭우, 폭설, 강풍'에 의해 실시간으로 발생합니다. 월평균 온도가 낮거나 비가 조금 더 왔다는 거시적 지표는 실시간 정시성 예측에 결정적인 힌트를 주기 어렵습니다.
*   **해결책**: 항공기 출발/도착 예정 시각(날짜 및 시간 단위)과 매핑되는 **시간대별 종관기상관측(ASOS) 실시간 날씨 데이터**를 공공데이터포털에서 가져와 결합해야 합니다.

#### 4. 도메인 특화 특성(Feature) 부족
*   **현황**: 현재 피처는 항공사코드, 공항, 출발대수, 평균 날씨 등 매우 기본적인 정보 위주입니다.
*   **문제점**: 항공기 지연의 가장 큰 병목인 **'이전 항공편의 지연 전파(A/C 접속 지연)'**, 활주로의 풍향 변화인 **'윈드시어(Wind Shear)'**, 또는 공항 자체의 **'시간대별 슬롯(Slot) 혼잡도'** 정보가 누락되어 있습니다.
*   **해결책**: 시간별 정체 지수 계산, 직전 항공편 지연 여부 플래그 생성 등의 Feature Engineering 작업이 추가되어야 합니다.

In [ ]:
from google.colab import userdata
import requests
import pandas as pd

# =====================================================================
# 🔑 [보안 적용] 코드창에 내 진짜 인증키 글자는 한 글자도 남기지 않습니다!
# =====================================================================
# 코랩 시스템이 열쇠(Secrets) 메뉴에 숨겨둔 키를 알아서 찾아와 변수에 넣습니다.
try:
    encoding_service_key = userdata.get('PUBLIC_DATA_KEY')
except Exception:
    print("⚠️ 코랩 좌측 '열쇠(Secrets)' 메뉴에서 PUBLIC_DATA_KEY를 등록해 주세요!")
    encoding_service_key = ""

# 한국공항공사 실시간 항공기 운항정보 오픈 API 주소
url = 'http://odcloud.kr'

params = {
    'serviceKey': encoding_service_key,  # 숨겨진 비밀 키가 안전하게 대입됩니다.
    'page': '1',
    'perPage': '10',  # 테스트용으로 10개만 요청
    'returnType': 'JSON'
}

response = requests.get(url, params=params)

if response.status_code == 200:
    print("✅ 보안 연결 성공! 팀원들에게 이 상태로 공유해도 안전합니다.")
    # 샘플 데이터가 정상적으로 수집되었는지 확인
    df_sample = pd.DataFrame(response.json().get('data', []))
    display(df_sample.head(2))
else:
    print(f"❌ 연결 실패 (오류코드: {response.status_code})")
    print("열쇠(Secrets) 메뉴의 Name이 'PUBLIC_DATA_KEY'인지, Value에 공백이 없는지 확인하세요.")

In [56]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from google.colab import userdata

# 🔑 [보안 조치] 하드코딩된 API 키를 제거하고 Colab Secrets(🔑) 환경변수로부터 안전하게 가져옵니다.
try:
    encoding_service_key = userdata.get('PUBLIC_DATA_KEY')
except Exception:
    print("⚠️ 코랩 좌측 '열쇠(Secrets)' 메뉴에서 PUBLIC_DATA_KEY를 등록해 주세요!")
    encoding_service_key = ''

# 한국공항공사 실시간 항공기 운항정보(출발/도착) 오픈 API 정확한 엔드포인트 URL
url = 'http://openapi.airport.co.kr/service/rest/FlightStatusList/getFlightStatusList'

all_flights = []
# 예시로 최근 일자 데이터 수집 (YYYYMMDD 형식)
target_date = '20240101'

params = {
    'serviceKey': requests.utils.unquote(encoding_service_key) if encoding_service_key else '', # URL 인코딩 중복 방지 처리
    'schDate': target_date,                # 검색일자
    'schAirCode': 'GMP',                   # 김포공항 기준
    'numOfRows': '100',                    # 한 번에 불러올 데이터 수
    'pageNo': '1'
}

print(f"🚀 [{target_date} 김포공항] 실시간 항공기 운항 데이터 API 수집을 시도합니다...")

if not encoding_service_key:
    print("❌ API 키가 등록되지 않아 요청을 보낼 수 없습니다. 코랩 Secrets 메뉴를 먼저 확인해 주세요.")
else:
    try:
        response = requests.get(url, params=params, timeout=15)
        if response.status_code == 200:
            # 공공데이터포털 API는 보통 XML 형태로 결과를 반환하므로, XML 파싱 처리를 수행합니다.
            root = ET.fromstring(response.content)
            items = root.findall('.//item')

            for item in items:
                row = {}
                for child in item:
                    row[child.tag] = child.text
                all_flights.append(row)

            if all_flights:
                df_result = pd.DataFrame(all_flights)
                display(df_result.head())
                df_result.to_csv('realtime_flight_data.csv', index=False, encoding='utf-8-sig')
                print(f"✅ 수집 성공! 총 {len(df_result)}건의 데이터가 'realtime_flight_data.csv' 파일로 저장되었습니다.")
            else:
                print("⚠️ API 호출은 성공했으나 반환된 항공 데이터가 없습니다. (인증키 권한 혹은 파라미터 확인 필요)")
                print(f"응답 내용 일부: {response.text[:300]}")
        else:
            print(f"❌ API 호출 실패! HTTP 상태 코드: {response.status_code}")
    except Exception as e:
        print(f"❌ 네트워크 및 연동 에러가 발생했습니다: {e}")

⚠️ 코랩 좌측 '열쇠(Secrets)' 메뉴에서 PUBLIC_DATA_KEY를 등록해 주세요!
🚀 [20240101 김포공항] 실시간 항공기 운항 데이터 API 수집을 시도합니다...
❌ API 키가 등록되지 않아 요청을 보낼 수 없습니다. 코랩 Secrets 메뉴를 먼저 확인해 주세요.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

print("🔄 [Colab 네트워크 우회] 실제 항공기 운항 이력을 모방한 10,000건의 대규모 합성 데이터셋을 빌드 중입니다...")

np.random.seed(42)
n_samples = 10000

# 실제 '한국공항공사_항공기운항정보.csv' 컬럼 양식에 기반한 리얼한 데이터 구성
airlines = ['ABL', 'TWB', 'AAR', 'KAL', 'JJA', 'JNA', 'EA_']
airports = ['GMP', 'CJU', 'PUS', 'TAE', 'USN', 'MWX', 'RSU', 'YNY']
flight_nums = [f'{i}' for i in range(1000, 9999)]

data_synthetic = pd.DataFrame({
    'CRT_SN': [f'2024_{i}' for i in range(n_samples)],
    'DT': ['2024/01/01 00:00:00'] * n_samples,
    'ARPR_CD_DPTRE_ARVL': np.random.choice(airports, n_samples),
    'ARMN_CD': np.random.choice(airlines, n_samples),
    'FLIGHT_NM': np.random.choice(flight_nums, n_samples),
    'PRNMNT_TM': [f'{np.random.randint(6, 23):02d}{np.random.randint(0, 60):02d}00' for _ in range(n_samples)],
    'ARPR_CD_ARVL_DPTRE': np.random.choice(airports, n_samples),
    'ARVL_DPTRE_SE': np.random.choice(['D', 'A'], n_samples), # Departure, Arrival
    'KOMA': np.random.choice(['Y', 'N'], n_samples, p=[0.9, 0.1])
})

# 시간 가공
def parse_time_to_min(t):
    try:
        return int(t[:2]) * 60 + int(t[2:4])
    except:
        return 720

data_synthetic['scheduled_min'] = data_synthetic['PRNMNT_TM'].apply(parse_time_to_min)
data_synthetic['scheduled_hour'] = data_synthetic['PRNMNT_TM'].apply(lambda x: int(x[:2]))

# 현실적인 지연(delay_minutes) 매핑 룰 (기상 악화 및 슬롯 혼잡 시나리오 결합)
data_synthetic['hourly_wind_speed'] = np.random.exponential(scale=2.5, size=n_samples)
data_synthetic['hourly_visibility'] = np.random.normal(loc=15000, scale=3000, size=n_samples).clip(100, 20000)

# 지연 시간(분) 조건부 생성
def generate_delay(row):
    base_delay = np.random.exponential(scale=8.0) # 기본 지연 시간
    if row['hourly_wind_speed'] > 10.0: # 강풍
        base_delay += np.random.exponential(scale=35.0)
    if row['hourly_visibility'] < 1500: # 저시정
        base_delay += np.random.exponential(scale=45.0)
    if row['ARPR_CD_DPTRE_ARVL'] == 'CJU' and np.random.rand() < 0.3: # 제주공항 혼잡 가중치
        base_delay += np.random.exponential(scale=15.0)
    return int(round(base_delay))

data_synthetic['delay_minutes'] = data_synthetic.apply(generate_delay, axis=1)
data_synthetic['is_delayed'] = (data_synthetic['delay_minutes'] >= 30).astype(int)

print(f"✅ 생성 완료: 총 {len(data_synthetic)}행 / 지연 발생 수 {data_synthetic['is_delayed'].sum()}건 ({data_synthetic['is_delayed'].mean()*100:.2f}%)")

# --- 3. 머신러닝 모델 피처 구성 및 학습 ---
features = ['ARMN_CD', 'ARVL_DPTRE_SE', 'scheduled_hour', 'hourly_wind_speed', 'hourly_visibility']
X_synthetic = pd.get_dummies(data_synthetic[features], drop_first=True)
y_synthetic = data_synthetic['is_delayed']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_synthetic, y_synthetic, test_size=0.2, random_state=42, stratify=y_synthetic
)

clf_synth = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight='balanced')
clf_synth.fit(X_train_s, y_train_s)

y_pred_s = clf_synth.predict(X_test_s)
print("\n=== 대용량 지연(정시성) 예측 시뮬레이션 평가 결과 ===")
print(f"정확도 (Accuracy): {accuracy_score(y_test_s, y_pred_s)*100:.2f}%")
print(classification_report(y_test_s, y_pred_s))

### 11. 로컬 데이터 기반 가우시안 노이즈 주입 및 데이터 증강(Augmentation)

업로드된 `한국공항공사_항공기운항정보.csv` 및 매핑된 기상 데이터를 바탕으로 가우시안 노이즈(정규분포를 따르는 무작위 잡음)를 수치형 변수에 추가하여 데이터셋을 5,000건 수준으로 증강합니다.

In [ ]:
import pandas as pd
import numpy as np

# 1. 기존 학습용 통합 데이터 복제 (df_merged_ml 변수 활용)
# df_merged_ml에 이미 공항별 mean_temp, mean_rain 등이 결합되어 있습니다.
df_base = df_merged_ml.copy()

# 증강하고자 하는 목표 데이터 크기
target_size = 5000
current_size = len(df_base)

print(f"기본 데이터 크기: {current_size}건 -> 목표 데이터 크기: {target_size}건")

# 2. 복원 추출을 통한 부트스트래핑(Re-sampling) 수행
np.random.seed(42)
multiply_factor = int(np.ceil(target_size / current_size))
df_augmented = pd.concat([df_base] * multiply_factor, ignore_index=True).sample(n=target_size, random_state=42).reset_index(drop=True)

# 3. 가우시안 노이즈 주입 함수 정의
def add_gaussian_noise(series, std_ratio=0.05):
    """
    시리즈 데이터의 표준편차에 비례하는 가우시안 노이즈를 추가합니다.
    """
    std = series.std()
    if pd.isna(std) or std == 0:
        std = series.mean() * 0.1 if series.mean() != 0 else 1.0

    noise = np.random.normal(loc=0.0, scale=std * std_ratio, size=len(series))
    return series + noise

# 4. 핵심 수치형 피처들에 가우시안 노이즈 반영
# 예정 시각(분), 평균 기온, 평균 강수량 변수에 미세한 잡음 주입
df_augmented['scheduled_min_noisy'] = add_gaussian_noise(df_augmented['scheduled_min'], std_ratio=0.03).round().astype(int)
df_augmented['mean_temp_noisy'] = add_gaussian_noise(df_augmented['mean_temp'], std_ratio=0.05)
df_augmented['mean_rain_noisy'] = add_gaussian_noise(df_augmented['mean_rain'], std_ratio=0.05)

# 예정 시각 범위를 하루(0 ~ 1440분) 안으로 제한(Clip)
df_augmented['scheduled_min_noisy'] = df_augmented['scheduled_min_noisy'].clip(0, 1439)
df_augmented['scheduled_hour_noisy'] = (df_augmented['scheduled_min_noisy'] // 60).astype(int)

# 5. 가우시안 노이즈 반영 결과 검증 비교
print("\n=== [가우시안 노이즈 주입 전/후 기술통계 비교] ===")
comparison_df = pd.DataFrame({
    '원본 예정시간(분) 평균': [df_base['scheduled_min'].mean()],
    '노이즈 예정시간(분) 평균': [df_augmented['scheduled_min_noisy'].mean()],
    '원본 평균기온 평균': [df_base['mean_temp'].mean()],
    '노이즈 평균기온 평균': [df_augmented['mean_temp_noisy'].mean()],
    '원본 강수량 평균': [df_base['mean_rain'].mean()],
    '노이즈 강수량 평균': [df_augmented['mean_rain_noisy'].mean()]
})
display(comparison_df)

print(f"✔ 가우시안 노이즈 증강 완료! 데이터셋 크기: {df_augmented.shape}")
display(df_augmented[['공항한글명', 'scheduled_min', 'scheduled_min_noisy', 'mean_temp', 'mean_temp_noisy', 'is_delayed']].head(10))

### 12. 증강된 데이터셋 기반 머신러닝 모델 학습 및 평가

가우시안 노이즈 주입을 통해 5,000건으로 늘어난 `df_augmented` 데이터를 피처로 정제하여 지연 분류 모델을 학습합니다.
- 입력 피처: `ARMN_CD`, `KOMA`, `ARVL_DPTRE_SE`, `scheduled_hour_noisy` (노이즈 반영), `mean_temp_noisy` (노이즈 반영), `mean_rain_noisy` (노이즈 반영)
- 타겟 변수: `is_delayed`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. 피처 및 타겟 설정 (노이즈가 반영된 증강 변수 활용)
feature_cols_augmented = [
    'ARMN_CD', 'KOMA', 'ARVL_DPTRE_SE',
    'scheduled_hour_noisy', 'mean_temp_noisy', 'mean_rain_noisy'
]

X_aug = pd.get_dummies(df_augmented[feature_cols_augmented], drop_first=True)
y_aug = df_augmented['is_delayed']

# 2. 데이터 분할 (Train 80% / Test 20%, 층화 추출 적용)
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_aug, y_aug, test_size=0.2, random_state=42, stratify=y_aug
)

# 3. 모델 정의 및 학습 (불균형 데이터 방지를 위해 class_weight 설정)
aug_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight='balanced')
aug_model.fit(X_train_a, y_train_a)

# 4. 성능 예측 및 검증
y_pred_a = aug_model.predict(X_test_a)
acc_aug = accuracy_score(y_test_a, y_pred_a)

print("=== [증강 데이터셋 학습] 지연 예측 분류 모델 결과 ===")
print(f"모델 예측 정확도 (Accuracy): {acc_aug * 100:.2f}%")
print("\n[정밀 성능 평가 보고서]")
print(classification_report(y_test_a, y_pred_a, zero_division=0))

# 5. 피처 중요도 시각화
importances_aug = aug_model.feature_importances_
indices_aug = np.argsort(importances_aug)[::-1]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances_aug[indices_aug][:10], y=X_aug.columns[indices_aug][:10], palette='magma', hue=X_aug.columns[indices_aug][:10], legend=False)
plt.title("증강 데이터셋 기반 중요 영향 요인 (Feature Importance)", fontsize=13)
plt.xlabel("중요도 (Importance)")
plt.ylabel("변수 (Features)")
plt.tight_layout()
plt.show()

### 13. SMOTE 기법 및 K-Fold 교차 검증을 통한 통계/모델 신뢰도 극대화

- **SMOTE**: 데이터 공간에서 KNN(K-Nearest Neighbors)을 기반으로 경계선 상에 새로운 데이터를 합성하여 클래스 불균형을 인위적이지 않게 해소합니다.
- **K-Fold Cross-Validation**: 데이터를 5개 구역으로 나누어 모든 데이터가 한 번씩 검증 세트로 사용되게 함으로써, 적은 데이터로도 매우 정밀하고 편향 없는 통계 평가를 내릴 수 있게 합니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# 1. 원본 통합 데이터셋 준비
X_raw = pd.get_dummies(df_merged_ml[feature_cols_weather], drop_first=True)
y_raw = df_merged_ml['is_delayed']

# 2. Stratified 5-Fold 교차 검증 객체 정의
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_list, recall_list, f1_list = [], [], []

print("=== SMOTE + Stratified 5-Fold Cross-Validation 시작 ===")
for fold, (train_idx, test_idx) in enumerate(skf.split(X_raw, y_raw), 1):
    X_train, X_test = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
    y_train, y_test = y_raw.iloc[train_idx], y_raw.iloc[test_idx]

    # 학습 데이터에 대해서만 SMOTE를 적용하여 데이터 균형 증강 (Over-sampling)
    # (테스트 데이터는 오염되지 않도록 유지)
    smote = SMOTE(k_neighbors=2, random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    # 모델 정의 및 학습
    fold_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
    fold_model.fit(X_train_res, y_train_res)

    # 예측 및 평가
    y_pred = fold_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    acc_list.append(acc)
    recall_list.append(rec)
    f1_list.append(f1)

    print(f"Fold {fold} - 증강후 학습데이터 크기: {len(X_train_res)} | 검증 정확도: {acc*100:.2f}% | 지연 재현율(Recall): {rec*100:.2f}% | F1-Score: {f1:.4f}")

print("\n=== 교차 검증 종합 통계 성적 ===")
print(f"평균 검증 정확도 (Mean Accuracy): {np.mean(acc_list)*100:.2f}%")
print(f"평균 지연 재현율 (Mean Recall): {np.mean(recall_list)*100:.2f}%")
print(f"평균 F1-Score (Mean F1): {np.mean(f1_list):.4f}")

### 14. SMOTE 파이프라인 기반 하이퍼파라미터 튜닝 (GridSearchCV)

데이터 누수(Data Leakage)를 원천 차단하기 위해 `imblearn.pipeline.Pipeline`을 구축하여 교차 검증 과정 안에서만 SMOTE가 올바르게 수행되도록 설계합니다. 그 후 `GridSearchCV`를 활용하여 `RandomForestClassifier`의 최적 하이퍼파라미터 조합을 탐색하고 정밀 분석합니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, make_scorer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

# 1. 원본 데이터셋 및 교차 검증 정의
X_raw = pd.get_dummies(df_merged_ml[feature_cols_weather], drop_first=True)
y_raw = df_merged_ml['is_delayed']
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. SMOTE + 임베디드 분류기 파이프라인 구축 (imblearn 전용 Pipeline 사용)
pipeline = ImbPipeline([
    ('smote', SMOTE(k_neighbors=2, random_state=42)),
    ('rf', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# 3. 하이퍼파라미터 탐색 격자(Grid) 설정
param_grid = {
    'rf__n_estimators': [50, 100, 150],
    'rf__max_depth': [4, 6, 8, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

# 4. 평가지표 설정 (지연 클래스의 F1-Score를 최적화 목표로 지정)
f1_scorer = make_scorer(f1_score, zero_division=0)

# 5. GridSearchCV 객체 생성 및 탐색 시작
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_stratified,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=1
)

print("🔄 SMOTE 파이프라인 하이퍼파라미터 튜닝(Grid Search) 진행 중...")
grid_search.fit(X_raw, y_raw)

# 6. 최적 조합 및 최종 성능 출력
print("\n=== 최적의 하이퍼파라미터 조합 ===")
for param, value in grid_search.best_params_.items():
    print(f" - {param.replace('rf__', '')}: {value}")

print(f"\n🏆 탐색된 최적 평균 F1-Score: {grid_search.best_score_:.4f}")

# 최적 파이프라인 모델 추출 및 교차 검증 최종 리포트 출력
best_pipeline = grid_search.best_estimator_
y_pred_cv = []
y_true_cv = []

for train_idx, test_idx in cv_stratified.split(X_raw, y_raw):
    X_tr, X_te = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
    y_tr, y_te = y_raw.iloc[train_idx], y_raw.iloc[test_idx]

    # 최적 파라미터가 내장된 파이프라인으로 적합 및 예측
    best_pipeline.fit(X_tr, y_tr)
    preds = best_pipeline.predict(X_te)

    y_pred_cv.extend(preds)
    y_true_cv.extend(y_te)

print("\n=== 최적 하이퍼파라미터 기반 OOF(Out-of-Fold) 전체 검증 성적 ===")
print(classification_report(y_true_cv, y_pred_cv, zero_division=0))

### 15. 최적 하이퍼파라미터 기반 최종 모델 학습 및 변수 중요도 분석

GridSearchCV에서 도출된 최적 파라미터 조합으로 전체 학습 데이터를 활용해 최종 파이프라인 모델을 학습시키고, 모델의 의사결정에 가장 크게 기여한 피처(Feature Importances)들을 상세히 분석합니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report

# 1. 최적의 파이프라인으로 전체 데이터 학습
print("🔄 최적 파라미터 파이프라인으로 최종 모델 학습 중...")
best_pipeline.fit(X_raw, y_raw)

# 2. 전체 데이터에 대한 예측 및 최종 리포트 출력
y_pred_final = best_pipeline.predict(X_raw)
print("\n=== [최종 모델] 전체 데이터 기반 예측 성능 리포트 ===")
print(classification_report(y_raw, y_pred_final, zero_division=0))

# 3. Random Forest 분류기의 피처 중요도 추출
rf_model = best_pipeline.named_steps['rf']
importances_final = rf_model.feature_importances_
indices_final = np.argsort(importances_final)[::-1]

# 4. 피처 중요도 시각화 (한글 깨짐 방지를 위해 깔끔한 스타일 적용)
slt_features = X_raw.columns[indices_final][:10]
slt_importances = importances_final[indices_final][:10]

plt.figure(figsize=(10, 6))
sns.barplot(x=slt_importances, y=slt_features, palette='viridis', hue=slt_features, legend=False)
plt.title("Final Model Feature Importances (Top 10)", fontsize=14)
plt.xlabel("Importance", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.tight_layout()
plt.show()